# LADI-v2 EfficientNet-B0 Full-Streaming Independent Notebook

This is a **fully independent Kaggle GPU P100 notebook** for the **EfficientNet-B0 CNN Baseline**.

It uses the LADI-v2 Hugging Face dataset in **streaming mode**, so the full dataset is **not downloaded into `/kaggle/working`**. Only small training outputs such as checkpoints, metrics, and CSV prediction files are saved.

## Model fixed in this notebook

```python
RUN_MODEL = "efficientnet_b0"
```

## Recommended Kaggle settings

```text
Accelerator: GPU P100
Internet: ON
Persistence: ON
```

Run the notebook from the first cell. If the environment setup cell restarts the kernel, run the notebook again from the top.


In [3]:
# ============================================================
# 0. Stable Kaggle P100 Environment Setup
# ============================================================
# This cell pins compatible versions to avoid common Kaggle errors:
# - CUDA: "no kernel image is available for execution on the device"
# - Pillow: cannot import _Ink from PIL._typing
# - NumPy: cannot import _center from numpy._core.umath

import os
import sys
import subprocess
from pathlib import Path

# Keep Hugging Face/cache files out of /kaggle/working output space.
os.environ["HF_HOME"] = "/kaggle/temp/hf_home"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf_datasets"
os.environ["HF_HUB_CACHE"] = "/kaggle/temp/hf_hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/transformers"
os.environ["TMPDIR"] = "/kaggle/temp"

for p in [
    os.environ["HF_HOME"],
    os.environ["HF_DATASETS_CACHE"],
    os.environ["HF_HUB_CACHE"],
    os.environ["TRANSFORMERS_CACHE"],
    os.environ["TMPDIR"],
]:
    Path(p).mkdir(parents=True, exist_ok=True)

ENV_MARKER = Path("/kaggle/temp/ladi_v2_final_streaming_env_ready_v1")

if not ENV_MARKER.exists():
    print("Installing stable Kaggle P100 environment. This may take a few minutes...")
    
    # Remove pip cache to protect disk space.
    subprocess.run([sys.executable, "-m", "pip", "cache", "purge"], check=False)
    
    base_packages = [
        "numpy==1.26.4",
        "scipy==1.13.1",
        "pandas==2.2.2",
        "scikit-learn==1.5.2",
        "pillow==10.4.0",
        "tqdm==4.66.5",
        "matplotlib==3.9.2",
        "datasets==2.21.0",
        "huggingface_hub==0.24.7",
    ]
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        *base_packages
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
        "torch==2.4.1", "torchvision==0.19.1",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ])
    
    ENV_MARKER.write_text("ready")
    print("\nEnvironment installed. Restarting kernel now.")
    print("After restart, run this notebook again from the first cell.")
    
    # Restart kernel cleanly after binary package changes.
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("Stable environment marker found. Skipping installation.")


Stable environment marker found. Skipping installation.


In [4]:
# ============================================================
# 1. Imports, Version Check, and CUDA Sanity Check
# ============================================================

import os
import io
import gc
import json
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
)

from datasets import load_dataset
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    x = torch.randn(2, 3, device="cuda")
    y = (x * 2).sum()
    y.backward if hasattr(y, "backward") else None
    print("CUDA sanity check passed.")
else:
    print("WARNING: CUDA is not available. Training will be slow on CPU.")


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
NumPy: 1.26.4
Pandas: 2.2.2
Torch: 2.4.1+cu118
Torchvision: 0.19.1+cu118
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Compute capability: (6, 0)
CUDA sanity check passed.


In [5]:
# ============================================================
# 2. Configuration
# ============================================================

# Fixed model for this notebook: efficientnet_b0
RUN_MODEL = "efficientnet_b0"

SEED = 42
DATASET_NAME = "MITLL/LADI-v2-dataset"

# Use None to try the dataset default first. The default is the practical resized v2a-style setup.
# If your Kaggle session requires explicit config, set DATASET_CONFIG = "v2a_resized".
DATASET_CONFIG = None

# Training settings
IMAGE_SIZE = 320
BATCH_SIZE = 16
NUM_EPOCHS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # keep 0 for Hugging Face streaming safety
USE_AMP = True

# Streaming settings
# None means use full split. For quick debugging, set TRAIN_LIMIT=512, VAL_LIMIT=128, TEST_LIMIT=128.
TRAIN_LIMIT = None
VAL_LIMIT = None
TEST_LIMIT = None

# Optional training-step cap. None means full epoch according to TRAIN_LIMIT/full stream.
MAX_TRAIN_STEPS_PER_EPOCH = None

# Shuffle buffer for streaming train split. Larger = better shuffle but more memory/network overhead.
SHUFFLE_BUFFER_SIZE = 2048

# Label-statistics scan. None means scan full train split to compute pos_weight and graph.
# For faster smoke test, set LABEL_SCAN_LIMIT = 2000.
LABEL_SCAN_LIMIT = None

# GCN settings
GCN_HIDDEN_DIM = 256
LABEL_EMBED_DIM = 256
GRAPH_TOPK = 5
DYNAMIC_GATE_INIT = 1.5  # sigmoid(1.5) ~ 0.82 static prior at start

# Output directory: only small model outputs go to /kaggle/working.
RUN_NAME = f"{RUN_MODEL}_streaming_full_seed{SEED}"
OUTPUT_DIR = Path("/kaggle/working/ladi_final_streaming_runs") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Run model:", RUN_MODEL)
print("Output directory:", OUTPUT_DIR)
print("Device:", DEVICE)


Run model: efficientnet_b0
Output directory: /kaggle/working/ladi_final_streaming_runs/efficientnet_b0_streaming_full_seed42
Device: cuda


In [6]:
# ============================================================
# 3. Streaming Dataset Loading
# ============================================================

def load_ladi_streaming():
    """Load LADI-v2 using Hugging Face streaming mode without saving full dataset to /kaggle/working."""
    attempts = []
    
    if DATASET_CONFIG is None:
        attempts.append({"path": DATASET_NAME, "streaming": True})
        attempts.append({"path": DATASET_NAME, "name": "v2a_resized", "streaming": True})
    else:
        attempts.append({"path": DATASET_NAME, "name": DATASET_CONFIG, "streaming": True})
        attempts.append({"path": DATASET_NAME, "streaming": True})
    
    last_err = None
    for kwargs in attempts:
        try:
            print("Trying load_dataset with:", kwargs)
            if "name" in kwargs:
                ds = load_dataset(kwargs["path"], kwargs["name"], streaming=kwargs["streaming"])
            else:
                ds = load_dataset(kwargs["path"], streaming=kwargs["streaming"])
            print("Loaded streaming dataset:", ds)
            return ds
        except Exception as e:
            last_err = e
            print("Failed attempt:", repr(e))
    
    raise RuntimeError(f"Could not load LADI-v2 streaming dataset. Last error: {last_err}")

stream_ds = load_ladi_streaming()
print("Available splits:", list(stream_ds.keys()))


Trying load_dataset with: {'path': 'MITLL/LADI-v2-dataset', 'streaming': True}


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loaded streaming dataset: IterableDatasetDict({
    train: IterableDataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        n_shards: 40
    })
    validation: IterableDataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        n_shards: 40
    })
    test: IterableDataset({
        features: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any'],
        n_shards: 40
    })
})
Available splits: ['train', 'v

In [7]:
# ============================================================
# 4. Label and Image Utilities
# ============================================================

LADI_V2A_LABELS = [
    "bridges_any",
    "buildings_any",
    "buildings_affected_or_greater",
    "buildings_minor_or_greater",
    "debris_any",
    "flooding_any",
    "flooding_structures",
    "roads_any",
    "roads_damage",
    "trees_any",
    "trees_damage",
    "water_any",
]

NON_LABEL_KEYS = {
    "image", "img", "pixel_values", "path", "filepath", "file", "filename", "image_id", "id",
    "url", "uri", "split", "event", "disaster", "date", "lat", "lon", "latitude", "longitude",
    "caption", "text", "metadata", "exif", "label", "labels"
}


def get_first_sample(split_name="train"):
    for sample in stream_ds[split_name]:
        return sample
    raise RuntimeError(f"Split {split_name} is empty.")

first_train_sample = get_first_sample("train")
print("First sample keys:", list(first_train_sample.keys()))


def infer_label_names(sample: Dict[str, Any]) -> List[str]:
    # Preferred: official LADI-v2a labels as individual binary columns.
    present = [c for c in LADI_V2A_LABELS if c in sample]
    if len(present) >= 6:
        return present
    
    # If there is a labels dictionary.
    if isinstance(sample.get("labels"), dict):
        keys = list(sample["labels"].keys())
        if keys:
            return keys
    
    # If there is a list-like labels field, keep generic names.
    if "labels" in sample and isinstance(sample["labels"], (list, tuple, np.ndarray)):
        return [f"label_{i}" for i in range(len(sample["labels"]))]
    
    # Last resort: infer binary numeric columns.
    inferred = []
    for k, v in sample.items():
        if k in NON_LABEL_KEYS:
            continue
        if isinstance(v, (int, float, bool, np.integer, np.floating)) and float(v) in [0.0, 1.0]:
            inferred.append(k)
    if inferred:
        return inferred
    
    raise RuntimeError(
        "Could not infer label names. Please inspect first_train_sample keys and manually set LABEL_NAMES."
    )

LABEL_NAMES = infer_label_names(first_train_sample)
NUM_LABELS = len(LABEL_NAMES)
print("Detected labels:")
for i, name in enumerate(LABEL_NAMES):
    print(f"{i:02d}: {name}")


def get_label_vector(sample: Dict[str, Any], label_names: List[str] = LABEL_NAMES) -> np.ndarray:
    if all(k in sample for k in label_names):
        return np.array([float(sample[k]) for k in label_names], dtype=np.float32)
    
    if isinstance(sample.get("labels"), dict):
        return np.array([float(sample["labels"].get(k, 0.0)) for k in label_names], dtype=np.float32)
    
    if "labels" in sample and isinstance(sample["labels"], (list, tuple, np.ndarray)):
        arr = np.array(sample["labels"], dtype=np.float32)
        return arr[:len(label_names)]
    
    raise RuntimeError("Could not extract label vector from sample.")


def get_image_from_sample(sample: Dict[str, Any]) -> Image.Image:
    # Common Hugging Face image column.
    for key in ["image", "img"]:
        if key in sample:
            obj = sample[key]
            if isinstance(obj, Image.Image):
                return obj.convert("RGB")
            if isinstance(obj, dict):
                if obj.get("bytes") is not None:
                    return Image.open(io.BytesIO(obj["bytes"])).convert("RGB")
                if obj.get("path") is not None:
                    return Image.open(obj["path"]).convert("RGB")
            if isinstance(obj, (str, Path)):
                return Image.open(obj).convert("RGB")
    
    # Some datasets use path-like fields.
    for key in ["path", "filepath", "file", "filename"]:
        if key in sample and sample[key]:
            return Image.open(sample[key]).convert("RGB")
    
    raise RuntimeError("Could not find image in sample.")


def get_sample_id(sample: Dict[str, Any], fallback_idx: int) -> str:
    for key in ["image_id", "id", "filename", "file", "path", "filepath", "url"]:
        if key in sample and sample[key] is not None:
            return str(sample[key])
    return f"sample_{fallback_idx}"

# Quick check
_y = get_label_vector(first_train_sample)
_img = get_image_from_sample(first_train_sample)
print("Image size:", _img.size)
print("Label vector shape:", _y.shape, "positive labels:", int(_y.sum()))


First sample keys: ['image', 'bridges_any', 'buildings_any', 'buildings_affected_or_greater', 'buildings_minor_or_greater', 'debris_any', 'flooding_any', 'flooding_structures', 'roads_any', 'roads_damage', 'trees_any', 'trees_damage', 'water_any']
Detected labels:
00: bridges_any
01: buildings_any
02: buildings_affected_or_greater
03: buildings_minor_or_greater
04: debris_any
05: flooding_any
06: flooding_structures
07: roads_any
08: roads_damage
09: trees_any
10: trees_damage
11: water_any
Image size: (1800, 1200)
Label vector shape: (12,) positive labels: 3


In [8]:
# ============================================================
# 5. Transforms and Streaming PyTorch Dataset
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class LADIStreamingTorchDataset(IterableDataset):
    def __init__(self, hf_streaming_dataset, split: str, transform, label_names: List[str],
                 limit: Optional[int] = None, shuffle: bool = False, buffer_size: int = 2048,
                 seed: int = 42):
        super().__init__()
        self.hf_streaming_dataset = hf_streaming_dataset
        self.split = split
        self.transform = transform
        self.label_names = label_names
        self.limit = limit
        self.shuffle = shuffle
        self.buffer_size = buffer_size
        self.seed = seed

    def __iter__(self):
        stream = self.hf_streaming_dataset[self.split]
        if self.shuffle:
            stream = stream.shuffle(buffer_size=self.buffer_size, seed=self.seed)
        yielded = 0
        for idx, sample in enumerate(stream):
            if self.limit is not None and yielded >= self.limit:
                break
            try:
                img = get_image_from_sample(sample)
                x = self.transform(img)
                y = torch.tensor(get_label_vector(sample, self.label_names), dtype=torch.float32)
                sid = get_sample_id(sample, idx)
                yielded += 1
                yield x, y, sid
            except Exception as e:
                # Skip rare broken images/samples without killing training.
                continue


def make_loader(split: str, train: bool, limit: Optional[int], seed: int) -> DataLoader:
    ds_torch = LADIStreamingTorchDataset(
        stream_ds,
        split=split,
        transform=train_transform if train else val_transform,
        label_names=LABEL_NAMES,
        limit=limit,
        shuffle=train,
        buffer_size=SHUFFLE_BUFFER_SIZE,
        seed=seed,
    )
    return DataLoader(
        ds_torch,
        batch_size=BATCH_SIZE,
        shuffle=False,  # shuffling handled by HF streaming shuffle
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

print("Streaming PyTorch dataset ready.")


Streaming PyTorch dataset ready.


In [9]:
# ============================================================
# 6. Label Statistics, pos_weight, and PMI Graph Construction
# ============================================================

@torch.no_grad()
def scan_label_statistics(limit: Optional[int] = LABEL_SCAN_LIMIT):
    counts = np.zeros(NUM_LABELS, dtype=np.float64)
    cooc = np.zeros((NUM_LABELS, NUM_LABELS), dtype=np.float64)
    n = 0
    stream = stream_ds["train"]
    pbar = tqdm(stream, desc="Scanning train labels", total=limit if limit is not None else None)
    for sample in pbar:
        if limit is not None and n >= limit:
            break
        try:
            y = get_label_vector(sample, LABEL_NAMES).astype(np.float64)
            y = (y > 0.5).astype(np.float64)
            counts += y
            cooc += np.outer(y, y)
            n += 1
            if n % 1000 == 0:
                pbar.set_postfix({"seen": n})
        except Exception:
            continue
    return n, counts, cooc

n_scan, label_counts, label_cooc = scan_label_statistics()
print("Scanned samples:", n_scan)
print("Label prevalence:")
for name, count in zip(LABEL_NAMES, label_counts):
    print(f"{name:35s}: {int(count):5d} ({count / max(n_scan, 1):.4f})")

# Positive-class weighting for BCEWithLogitsLoss.
pos = label_counts
neg = max(n_scan, 1) - pos
pos_weight_np = neg / np.maximum(pos, 1.0)
pos_weight_np = np.clip(pos_weight_np, 1.0, 20.0)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=DEVICE)
print("pos_weight:", pos_weight.detach().cpu().numpy())


def build_pmi_graph(cooc: np.ndarray, counts: np.ndarray, n: int, topk: int = GRAPH_TOPK, eps: float = 1e-8) -> np.ndarray:
    p_i = counts / max(n, 1)
    p_ij = cooc / max(n, 1)
    expected = np.outer(p_i, p_i)
    pmi = np.log((p_ij + eps) / (expected + eps))
    pmi = np.maximum(pmi, 0.0)
    np.fill_diagonal(pmi, 1.0)
    
    # Top-k sparsification per row, keeping diagonal.
    A = np.zeros_like(pmi, dtype=np.float32)
    for i in range(pmi.shape[0]):
        row = pmi[i].copy()
        top_idx = np.argsort(row)[-topk:]
        A[i, top_idx] = row[top_idx]
        A[i, i] = 1.0
    
    # Symmetrise.
    A = np.maximum(A, A.T)
    
    # Add self-loop and symmetric normalisation.
    np.fill_diagonal(A, np.maximum(np.diag(A), 1.0))
    degree = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(degree, eps)))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt
    return A_norm.astype(np.float32)

A_PMI = build_pmi_graph(label_cooc, label_counts, n_scan)
A_PMI_TENSOR = torch.tensor(A_PMI, dtype=torch.float32, device=DEVICE)
print("PMI graph shape:", A_PMI_TENSOR.shape)
print(pd.DataFrame(A_PMI, index=LABEL_NAMES, columns=LABEL_NAMES).round(3))


Scanning train labels: 0it [00:00, ?it/s]

Scanned samples: 8030
Label prevalence:
bridges_any                        :   575 (0.0716)
buildings_any                      :  4768 (0.5938)
buildings_affected_or_greater      :  1488 (0.1853)
buildings_minor_or_greater         :   476 (0.0593)
debris_any                         :   674 (0.0839)
flooding_any                       :  1455 (0.1812)
flooding_structures                :   443 (0.0552)
roads_any                          :  5288 (0.6585)
roads_damage                       :   360 (0.0448)
trees_any                          :  7263 (0.9045)
trees_damage                       :  1964 (0.2446)
water_any                          :  5288 (0.6585)
pos_weight: [12.965218   1.         4.3965054 15.869748  10.913946   4.5189004
 17.126411   1.        20.         1.         3.0885947  1.       ]
PMI graph shape: torch.Size([12, 12])
                               bridges_any  buildings_any  \
bridges_any                          0.256          0.000   
buildings_any                

In [10]:
# ============================================================
# 7. Model Definitions
# ============================================================

def build_cnn_feature_extractor(name: str):
    """Return feature extractor and feature dimension."""
    if name == "resnet18":
        try:
            weights = ResNet18_Weights.DEFAULT
            model = resnet18(weights=weights)
            print("Loaded pretrained ResNet18 weights.")
        except Exception as e:
            print("Could not load pretrained ResNet18 weights. Using random init.", repr(e))
            model = resnet18(weights=None)
        feat_dim = model.fc.in_features
        model.fc = nn.Identity()
        return model, feat_dim
    
    if name == "efficientnet_b0":
        try:
            weights = EfficientNet_B0_Weights.DEFAULT
            model = efficientnet_b0(weights=weights)
            print("Loaded pretrained EfficientNet-B0 weights.")
        except Exception as e:
            print("Could not load pretrained EfficientNet-B0 weights. Using random init.", repr(e))
            model = efficientnet_b0(weights=None)
        feat_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        return model, feat_dim
    
    raise ValueError(f"Unknown CNN feature extractor: {name}")

class CNNMultiLabelClassifier(nn.Module):
    def __init__(self, backbone_name: str, num_labels: int, dropout: float = 0.2):
        super().__init__()
        self.backbone, feat_dim = build_cnn_feature_extractor(backbone_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feat_dim, num_labels)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        return self.classifier(feat)

class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A):
        # X: [L, D], A: [L, L]
        return self.linear(A @ X)

class StaticGCNClassifier(nn.Module):
    def __init__(self, num_labels: int, A: torch.Tensor, backbone_name: str = "resnet18",
                 label_dim: int = LABEL_EMBED_DIM, hidden_dim: int = GCN_HIDDEN_DIM, dropout: float = 0.2):
        super().__init__()
        self.backbone, feat_dim = build_cnn_feature_extractor(backbone_name)
        self.register_buffer("A", A.clone().float())
        self.label_embeddings = nn.Parameter(torch.randn(num_labels, label_dim) * 0.02)
        self.gcn1 = GCNLayer(label_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, feat_dim)
        self.bias = nn.Parameter(torch.zeros(num_labels))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.dropout(feat)
        Z = F.relu(self.gcn1(self.label_embeddings, self.A))
        Z = self.dropout(Z)
        W = self.gcn2(Z, self.A)  # [L, feat_dim]
        logits = feat @ W.t() + self.bias
        return logits

class BatchGCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A):
        # X: [B, L, D], A: [B, L, L]
        return self.linear(torch.bmm(A, X))

class DynamicGCNClassifier(nn.Module):
    def __init__(self, num_labels: int, A_static: torch.Tensor, backbone_name: str = "resnet18",
                 label_dim: int = LABEL_EMBED_DIM, hidden_dim: int = GCN_HIDDEN_DIM,
                 gate_init: float = DYNAMIC_GATE_INIT, dropout: float = 0.2):
        super().__init__()
        self.backbone, feat_dim = build_cnn_feature_extractor(backbone_name)
        self.num_labels = num_labels
        self.feat_dim = feat_dim
        self.label_dim = label_dim
        self.register_buffer("A_static", A_static.clone().float())
        
        self.label_embeddings = nn.Parameter(torch.randn(num_labels, label_dim) * 0.02)
        self.context_proj = nn.Linear(feat_dim, label_dim)
        self.node_gate = nn.Linear(feat_dim, label_dim)
        
        self.gcn1 = BatchGCNLayer(label_dim, hidden_dim)
        self.gcn2 = BatchGCNLayer(hidden_dim, feat_dim)
        self.bias = nn.Parameter(torch.zeros(num_labels))
        self.mix_gate = nn.Parameter(torch.tensor(float(gate_init)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        feat = self.backbone(x)  # [B, F]
        feat_d = self.dropout(feat)
        B = feat.shape[0]
        L = self.num_labels
        
        base_nodes = self.label_embeddings.unsqueeze(0).expand(B, L, self.label_dim)
        context = torch.tanh(self.context_proj(feat)).unsqueeze(1)  # [B, 1, D]
        gate = torch.sigmoid(self.node_gate(feat)).unsqueeze(1)     # [B, 1, D]
        nodes = base_nodes + gate * context
        
        # Dynamic adjacency from image-conditioned label nodes.
        sim = torch.bmm(nodes, nodes.transpose(1, 2)) / math.sqrt(self.label_dim)
        A_dyn = torch.softmax(sim, dim=-1)
        
        # Static prior expanded to batch.
        A_static = self.A_static.unsqueeze(0).expand(B, L, L)
        alpha = torch.sigmoid(self.mix_gate)  # alpha near 1 => more static prior
        A_mix = alpha * A_static + (1.0 - alpha) * A_dyn
        
        Z = F.relu(self.gcn1(nodes, A_mix))
        Z = self.dropout(Z)
        W = self.gcn2(Z, A_mix)  # [B, L, F]
        logits = torch.einsum("bf,blf->bl", feat_d, W) + self.bias
        return logits


def build_model(run_model: str) -> nn.Module:
    if run_model == "resnet18":
        return CNNMultiLabelClassifier("resnet18", NUM_LABELS)
    if run_model == "efficientnet_b0":
        return CNNMultiLabelClassifier("efficientnet_b0", NUM_LABELS)
    if run_model == "static_gcn_resnet18":
        return StaticGCNClassifier(NUM_LABELS, A_PMI_TENSOR, backbone_name="resnet18")
    if run_model == "dynamic_gcn_resnet18":
        return DynamicGCNClassifier(NUM_LABELS, A_PMI_TENSOR, backbone_name="resnet18")
    raise ValueError(f"Unsupported RUN_MODEL: {run_model}")

model = build_model(RUN_MODEL).to(DEVICE)
print(model.__class__.__name__)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 166MB/s]

Loaded pretrained EfficientNet-B0 weights.
CNNMultiLabelClassifier
Trainable parameters: 4022920


In [11]:
# ============================================================
# 8. Metrics and Threshold Tuning
# ============================================================

def safe_average_precision(y_true, y_prob):
    scores = []
    for j in range(y_true.shape[1]):
        if len(np.unique(y_true[:, j])) < 2:
            scores.append(np.nan)
        else:
            scores.append(average_precision_score(y_true[:, j], y_prob[:, j]))
    return np.array(scores, dtype=np.float64)


def tune_thresholds(y_true: np.ndarray, y_prob: np.ndarray, grid=None) -> np.ndarray:
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    for j in range(y_true.shape[1]):
        best_f1 = -1.0
        best_t = 0.5
        if len(np.unique(y_true[:, j])) < 2:
            thresholds[j] = 0.5
            continue
        for t in grid:
            pred = (y_prob[:, j] >= t).astype(int)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        thresholds[j] = best_t
    return thresholds


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, thresholds: Optional[np.ndarray] = None) -> Dict[str, Any]:
    if thresholds is None:
        thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    y_pred = (y_prob >= thresholds.reshape(1, -1)).astype(int)
    per_label_ap = safe_average_precision(y_true, y_prob)
    per_label_f1 = np.array([
        f1_score(y_true[:, j], y_pred[:, j], zero_division=0)
        for j in range(y_true.shape[1])
    ])
    metrics = {
        "macro_ap": float(np.nanmean(per_label_ap)),
        "micro_ap": float(average_precision_score(y_true.ravel(), y_prob.ravel())) if len(np.unique(y_true.ravel())) > 1 else float("nan"),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_label_ap": {LABEL_NAMES[i]: None if np.isnan(per_label_ap[i]) else float(per_label_ap[i]) for i in range(NUM_LABELS)},
        "per_label_f1": {LABEL_NAMES[i]: float(per_label_f1[i]) for i in range(NUM_LABELS)},
        "thresholds": {LABEL_NAMES[i]: float(thresholds[i]) for i in range(NUM_LABELS)},
    }
    return metrics


def save_predictions_csv(ids: List[str], y_true: np.ndarray, y_prob: np.ndarray, thresholds: np.ndarray, path: Path):
    data = {"sample_id": ids}
    y_pred = (y_prob >= thresholds.reshape(1, -1)).astype(int)
    for i, label in enumerate(LABEL_NAMES):
        data[f"true_{label}"] = y_true[:, i].astype(int)
        data[f"prob_{label}"] = y_prob[:, i]
        data[f"pred_{label}"] = y_pred[:, i].astype(int)
    pd.DataFrame(data).to_csv(path, index=False)

print("Metric utilities ready.")


Metric utilities ready.


In [12]:
# ============================================================
# 9. Training and Evaluation Functions
# ============================================================

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))


def train_one_epoch(epoch: int) -> float:
    model.train()
    loader = make_loader("train", train=True, limit=TRAIN_LIMIT, seed=SEED + epoch)
    total_loss = 0.0
    total_batches = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch} train")
    
    for step, (x, y, _) in enumerate(pbar):
        if MAX_TRAIN_STEPS_PER_EPOCH is not None and step >= MAX_TRAIN_STEPS_PER_EPOCH:
            break
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast("cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += float(loss.detach().cpu())
        total_batches += 1
        pbar.set_postfix({"loss": total_loss / max(total_batches, 1)})
        
    return total_loss / max(total_batches, 1)


@torch.no_grad()
def predict_split(split: str, limit: Optional[int]) -> Tuple[List[str], np.ndarray, np.ndarray]:
    model.eval()
    loader = make_loader(split, train=False, limit=limit, seed=SEED)
    all_ids, all_probs, all_targets = [], [], []
    for x, y, ids in tqdm(loader, desc=f"Predict {split}"):
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
            logits = model(x)
            probs = torch.sigmoid(logits)
        all_ids.extend(list(ids))
        all_probs.append(probs.detach().cpu().numpy())
        all_targets.append(y.numpy())
    if not all_probs:
        raise RuntimeError(f"No predictions collected for split={split}. Check dataset loading.")
    return all_ids, np.concatenate(all_targets, axis=0), np.concatenate(all_probs, axis=0)

print("Training functions ready.")


Training functions ready.


In [13]:
# ============================================================
# 10. Train Model
# ============================================================

history = []
best_macro_ap = -1.0
best_path = OUTPUT_DIR / "best_model.pt"
best_thresholds = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(epoch)
    
    val_ids, y_val, p_val = predict_split("validation", VAL_LIMIT)
    thresholds = tune_thresholds(y_val, p_val)
    val_metrics = compute_metrics(y_val, p_val, thresholds)
    
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_macro_ap": val_metrics["macro_ap"],
        "val_micro_ap": val_metrics["micro_ap"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_micro_f1": val_metrics["micro_f1"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / "history.csv", index=False)
    
    print(json.dumps(row, indent=2))
    
    if val_metrics["macro_ap"] > best_macro_ap:
        best_macro_ap = val_metrics["macro_ap"]
        best_thresholds = thresholds.copy()
        torch.save({
            "model_state_dict": model.state_dict(),
            "run_model": RUN_MODEL,
            "label_names": LABEL_NAMES,
            "thresholds": best_thresholds.tolist(),
            "val_metrics": val_metrics,
            "config": {
                "RUN_MODEL": RUN_MODEL,
                "IMAGE_SIZE": IMAGE_SIZE,
                "BATCH_SIZE": BATCH_SIZE,
                "NUM_EPOCHS": NUM_EPOCHS,
                "LEARNING_RATE": LEARNING_RATE,
                "WEIGHT_DECAY": WEIGHT_DECAY,
                "TRAIN_LIMIT": TRAIN_LIMIT,
                "VAL_LIMIT": VAL_LIMIT,
                "TEST_LIMIT": TEST_LIMIT,
                "LABEL_SCAN_LIMIT": LABEL_SCAN_LIMIT,
            }
        }, best_path)
        save_predictions_csv(val_ids, y_val, p_val, best_thresholds, OUTPUT_DIR / "val_predictions_best.csv")
        with open(OUTPUT_DIR / "val_metrics_best.json", "w") as f:
            json.dump(val_metrics, f, indent=2)
        print(f"New best model saved: {best_path} | val_macro_ap={best_macro_ap:.4f}")
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Training complete. Best val macro AP:", best_macro_ap)


Epoch 1 train: 0it [00:00, ?it/s]

Predict validation: 0it [00:00, ?it/s]

{
  "epoch": 1,
  "train_loss": 0.7007921899101174,
  "val_macro_ap": 0.7587176119314276,
  "val_micro_ap": 0.9032562261072622,
  "val_macro_f1": 0.7071312515200443,
  "val_micro_f1": 0.8467365967365967
}
New best model saved: /kaggle/working/ladi_final_streaming_runs/efficientnet_b0_streaming_full_seed42/best_model.pt | val_macro_ap=0.7587


Epoch 2 train: 0it [00:00, ?it/s]

Predict validation: 0it [00:00, ?it/s]

...

Predict validation: 0it [00:00, ?it/s]

{
  "epoch": 7,
  "train_loss": 0.3061959896815488,
  "val_macro_ap": 0.7940418791334684,
  "val_micro_ap": 0.9283009442428695,
  "val_macro_f1": 0.7467139170419884,
  "val_micro_f1": 0.8725475948263334
}


Epoch 8 train: 0it [00:00, ?it/s]

Predict validation: 0it [00:00, ?it/s]

{
  "epoch": 8,
  "train_loss": 0.2833567705465503,
  "val_macro_ap": 0.7989720129723002,
  "val_micro_ap": 0.9308787508444292,
  "val_macro_f1": 0.7494918330546207,
  "val_micro_f1": 0.86820593718629
}
Training complete. Best val macro AP: 0.8027234114592536


In [14]:
# ============================================================
# 11. Test Evaluation Using Best Checkpoint
# ============================================================

ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
best_thresholds = np.array(ckpt["thresholds"], dtype=np.float32)

# Validation summary from best checkpoint.
print("Best validation metrics:")
print(json.dumps(ckpt["val_metrics"], indent=2))

# Test evaluation.
test_ids, y_test, p_test = predict_split("test", TEST_LIMIT)
test_metrics = compute_metrics(y_test, p_test, best_thresholds)

save_predictions_csv(test_ids, y_test, p_test, best_thresholds, OUTPUT_DIR / "test_predictions.csv")
with open(OUTPUT_DIR / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

# Save per-label table.
per_label_rows = []
for label in LABEL_NAMES:
    per_label_rows.append({
        "label": label,
        "ap": test_metrics["per_label_ap"][label],
        "f1": test_metrics["per_label_f1"][label],
        "threshold": test_metrics["thresholds"][label],
    })
pd.DataFrame(per_label_rows).to_csv(OUTPUT_DIR / "test_per_label_metrics.csv", index=False)

# Save config.
config = {
    "RUN_MODEL": RUN_MODEL,
    "DATASET_NAME": DATASET_NAME,
    "DATASET_CONFIG": DATASET_CONFIG,
    "LABEL_NAMES": LABEL_NAMES,
    "IMAGE_SIZE": IMAGE_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_EPOCHS": NUM_EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "TRAIN_LIMIT": TRAIN_LIMIT,
    "VAL_LIMIT": VAL_LIMIT,
    "TEST_LIMIT": TEST_LIMIT,
    "LABEL_SCAN_LIMIT": LABEL_SCAN_LIMIT,
    "SHUFFLE_BUFFER_SIZE": SHUFFLE_BUFFER_SIZE,
}
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Test metrics:")
print(json.dumps({k: v for k, v in test_metrics.items() if not k.startswith("per_label") and k != "thresholds"}, indent=2))
print("\nSaved outputs to:", OUTPUT_DIR)
print("Files:", sorted([p.name for p in OUTPUT_DIR.iterdir()]))


Best validation metrics:
{
  "macro_ap": 0.8027234114592536,
  "micro_ap": 0.9264937322258573,
  "macro_f1": 0.7529590475655241,
  "micro_f1": 0.8702513150204558,
  "macro_precision": 0.7462056776216884,
  "macro_recall": 0.7661199021243843,
  "per_label_ap": {
    "bridges_any": 0.726541376323455,
    "buildings_any": 0.9873419183806118,
    "buildings_affected_or_greater": 0.8243206921322431,
    "buildings_minor_or_greater": 0.7294705882159951,
    "debris_any": 0.7172132861805903,
    "flooding_any": 0.8214025300812301,
    "flooding_structures": 0.7474732867969952,
    "roads_any": 0.9808492722844915,
    "roads_damage": 0.3691240405317017,
    "trees_any": 0.9932695135597414,
    "trees_damage": 0.7854210214254601,
    "water_any": 0.9502534115985283
  },
  "per_label_f1": {
    "bridges_any": 0.6746987951807228,
    "buildings_any": 0.9499036608863198,
    "buildings_affected_or_greater": 0.75,
    "buildings_minor_or_greater": 0.6417910447761194,
    "debris_any": 0.66321243523

Predict test: 0it [00:00, ?it/s]

Test metrics:
{
  "macro_ap": 0.5423058556791781,
  "micro_ap": 0.8712761051178833,
  "macro_f1": 0.5369775629197854,
  "micro_f1": 0.8260427263479145,
  "macro_precision": 0.526183211613516,
  "macro_recall": 0.5633535949738988
}

Saved outputs to: /kaggle/working/ladi_final_streaming_runs/efficientnet_b0_streaming_full_seed42
Files: ['best_model.pt', 'config.json', 'history.csv', 'test_metrics.json', 'test_per_label_metrics.csv', 'test_predictions.csv', 'val_metrics_best.json', 'val_predictions_best.csv']


In [15]:
# ============================================================
# 12. Optional: Display Final Summary Tables
# ============================================================

history_df = pd.read_csv(OUTPUT_DIR / "history.csv")
per_label_df = pd.read_csv(OUTPUT_DIR / "test_per_label_metrics.csv")

print("Training history:")
display(history_df)

print("Test per-label metrics:")
display(per_label_df.sort_values("ap", ascending=False))


Training history:


,epoch,train_loss,val_macro_ap,val_micro_ap,val_macro_f1,val_micro_f1
0,1,0.700792,0.758718,0.903256,0.707131,0.846737
1,2,0.548141,0.772382,0.915888,0.729513,0.859533
2,3,0.480672,0.783838,0.925382,0.737493,0.859278
3,4,0.415821,0.802723,0.926494,0.752959,0.870251
4,5,0.383362,0.801555,0.924652,0.753844,0.871316
5,6,0.344780,0.792039,0.926897,0.754927,0.868881
6,7,0.306196,0.794042,0.928301,0.746714,0.872548
7,8,0.283357,0.798972,0.930879,0.749492,0.868206


Test per-label metrics:


,label,ap,f1,threshold
9,trees_any,0.994194,0.949868,0.30
1,buildings_any,0.980772,0.906555,0.55
7,roads_any,0.966806,0.902761,0.40
11,water_any,0.814802,0.712195,0.40
5,flooding_any,0.489549,0.522613,0.75
2,buildings_affected_or_greater,0.438106,0.508475,0.70
0,bridges_any,0.413097,0.384615,0.95
3,buildings_minor_or_greater,0.410894,0.454545,0.90
4,debris_any,0.402108,0.421053,0.80
10,trees_damage,0.318263,0.326996,0.65


In [16]:
df=pd.read_csv('/kaggle/working/ladi_final_streaming_runs/efficientnet_b0_streaming_full_seed42/test_predictions.csv')
df

,sample_id,true_bridges_any,prob_bridges_any,pred_bridges_any,true_buildings_any,prob_buildings_any,pred_buildings_any,true_buildings_affected_or_greater,prob_buildings_affected_or_greater,pred_buildings_affected_or_greater,...,pred_roads_damage,true_trees_any,prob_trees_any,pred_trees_any,true_trees_damage,prob_trees_damage,pred_trees_damage,true_water_any,prob_water_any,pred_water_any
0,sample_0,0,0.032300,0,1,0.995000,1,1,0.47120,0,...,0,1,0.9890,1,1,0.5440,0,0,0.3066,0
1,sample_1,0,0.290800,0,1,0.999000,1,1,0.93160,1,...,0,1,0.9020,1,1,0.7207,1,0,0.1260,0
2,sample_2,0,0.027010,0,0,0.109500,0,0,0.01799,0,...,0,1,0.9956,1,1,0.3752,0,1,0.9805,1
3,sample_3,0,0.034240,0,1,0.999000,1,1,0.98140,1,...,0,1,0.9930,1,1,0.9100,1,0,0.1318,0
4,sample_4,0,0.039550,0,1,0.970700,1,1,0.44480,0,...,0,1,0.9766,1,1,0.4531,0,0,0.1528,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,sample_1044,0,0.005302,0,0,0.091740,0,0,0.06930,0,...,0,1,0.9830,1,1,0.8926,1,1,0.9940,1
1045,sample_1045,0,0.020770,0,1,0.854000,1,0,0.41850,0,...,0,1,0.9976,1,1,0.8550,1,1,0.9600,1
1046,sample_1046,0,0.001179,0,1,0.996000,1,0,0.14450,0,...,0,1,0.9970,1,0,0.1523,0,1,0.5586,1
1047,sample_1047,0,0.005753,0,0,0.007694,0,0,0.00582,0,...,0,1,0.9756,1,1,0.8890,1,1,0.9990,1


## Notebook Completion Notes

This notebook is fixed to:

```python
RUN_MODEL = "efficientnet_b0"
```

Outputs are saved under:

```text
/kaggle/working/ladi_final_streaming_runs/efficientnet_b0_streaming_full_seed42/
```

For a smoke test, use small limits in the configuration cell:

```python
TRAIN_LIMIT = 512
VAL_LIMIT = 128
TEST_LIMIT = 128
NUM_EPOCHS = 2
LABEL_SCAN_LIMIT = 1000
```

For a final streaming run, use:

```python
TRAIN_LIMIT = None
VAL_LIMIT = None
TEST_LIMIT = None
LABEL_SCAN_LIMIT = None
```
